<!-- Cell 1 -->
# AquaSelect Cross-Dataset Validation -- Notebook SA-2
# Sea Animals: Selective Prediction (SR, MCD, AquaSelect)

**Purpose**: Validate that AquaSelect framework generalizes to a second marine dataset.  
**Methods**: SR, MCD, AquaSelect (frozen backbone, binary BCE selection head).  
**Hardware**: Kaggle T4 GPU

In [ ]:
# Cell 2
!pip install -q timm

import os, copy, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from pathlib import Path

import timm
import cv2
from torchvision import transforms
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Configuration 
CFG = {
    "seeds": [9, 42, 2003],
    "batch_size": 32,
    "num_workers": 2,
    "image_size": 224,
    "mcd_forward_passes": 10,
    "mcd_dropout_rate": 0.3,
    "sel_epochs": 15,
    "sel_lr": 1e-3,
    "sel_patience": 5,
    "output_dir": "/kaggle/working",
}

PATHS = {
    "sa1_outputs": "/kaggle/input/datasets/abcd/aquaselect-on-another-dataset-nb1-models",
    "sea_animals_data": "/kaggle/input/datasets/vencerlanz09/sea-animals-image-dataste",
}

# Load SA-1 outputs 
sa1_data = torch.load(os.path.join(PATHS["sa1_outputs"], "sea_animals_logits.pth"),
                       map_location="cpu", weights_only=False)

label_names = sa1_data["label_names"]
num_classes = len(label_names)
all_image_paths = sa1_data["all_image_paths"]
all_labels = sa1_data["all_labels"]
train_indices = sa1_data["train_indices"]
val_indices = sa1_data["val_indices"]
test_indices = sa1_data["test_indices"]

print(f"Classes: {num_classes}")
print(f"Train: {len(train_indices)} | Val: {len(val_indices)} | Test: {len(test_indices)}")

# Verify logits
for seed in CFG["seeds"]:
    acc = (sa1_data["test_logits"][seed].argmax(1) == sa1_data["test_labels"]).float().mean() * 100
    print(f"Seed {seed} test acc: {acc:.2f}%")

In [ ]:
# Cell 3
# Dataset, dataloaders, model factory, utilities 

eval_transform = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(CFG["image_size"]),
    transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
train_aug_transform = transforms.Compose([
    transforms.RandomResizedCrop(CFG["image_size"], scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(), transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
raw_transform = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(CFG["image_size"])])


class SeaAnimalsDataset(Dataset):
    def __init__(self, image_paths, labels, indices, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.indices = indices
        self.transform = transform
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        image = Image.open(self.image_paths[real_idx]).convert("RGB")
        label = self.labels[real_idx]
        if self.transform:
            image = self.transform(image)
        return image, label


val_loader = DataLoader(SeaAnimalsDataset(all_image_paths, all_labels, val_indices, eval_transform),
                        batch_size=CFG["batch_size"], shuffle=False, num_workers=CFG["num_workers"], pin_memory=True)
test_loader = DataLoader(SeaAnimalsDataset(all_image_paths, all_labels, test_indices, eval_transform),
                         batch_size=CFG["batch_size"], shuffle=False, num_workers=CFG["num_workers"], pin_memory=True)
train_aug_loader = DataLoader(SeaAnimalsDataset(all_image_paths, all_labels, train_indices, train_aug_transform),
                              batch_size=CFG["batch_size"], shuffle=True, num_workers=CFG["num_workers"], pin_memory=True)


class BackboneClassifier(nn.Module):
    def __init__(self, backbone, classifier):
        super().__init__()
        self.backbone = backbone
        self.classifier = classifier
    def forward(self, x):
        features = self.backbone(x)
        logits = self.classifier(features)
        return logits, features


def load_trained_model(seed):
    ckpt_file = os.path.join(PATHS["sa1_outputs"], f"convnext_tiny_sea_animals_seed_{seed}.pth")
    ckpt = torch.load(ckpt_file, map_location="cpu", weights_only=False)
    backbone = timm.create_model("convnext_tiny", pretrained=False, num_classes=0)
    classifier = nn.Linear(768, num_classes)
    model = BackboneClassifier(backbone, classifier)
    model.load_state_dict(ckpt["model_state_dict"])
    return model


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False


# Selective prediction metrics 
def compute_risk_coverage_curve(predictions, targets, confidence_scores):
    sorted_indices = np.argsort(-confidence_scores)
    sorted_correct = (predictions == targets)[sorted_indices]
    n = len(sorted_correct)
    coverages, risks = [], []
    for k in range(1, n + 1):
        coverages.append(k / n)
        risks.append(1.0 - sorted_correct[:k].mean())
    aurcc = np.trapz(risks, coverages)
    return np.array(coverages), np.array(risks), aurcc

def coverage_at_accuracy(predictions, targets, confidence_scores, target_acc=0.95):
    sorted_indices = np.argsort(-confidence_scores)
    sorted_correct = (predictions == targets)[sorted_indices]
    n = len(sorted_correct)
    best_coverage = 0.0
    for k in range(1, n + 1):
        if sorted_correct[:k].mean() >= target_acc:
            best_coverage = k / n
    return best_coverage * 100

def evaluate_selection(preds, targets, conf, method_name=""):
    _, _, aurcc = compute_risk_coverage_curve(preds, targets, conf)
    cov95 = coverage_at_accuracy(preds, targets, conf, 0.95)
    cov99 = coverage_at_accuracy(preds, targets, conf, 0.99)
    return {"method": method_name, "aurcc": aurcc, "cov95": cov95, "cov99": cov99,
            "full_acc": (preds == targets).mean() * 100}


print(f"Val loader: {len(val_loader)} batches | Test loader: {len(test_loader)} batches | Train aug: {len(train_aug_loader)} batches")
print("All utilities defined.")

In [ ]:
# Cell 4
# Softmax Response (SR) + MC Dropout (MCD) 

# SR 
print("--- Softmax Response (SR) ---")
sr_results = {}
for seed in CFG["seeds"]:
    test_logits = sa1_data["test_logits"][seed]
    test_labels = sa1_data["test_labels"]
    probs = F.softmax(test_logits, dim=1)
    conf = probs.max(dim=1).values.numpy()
    preds = test_logits.argmax(dim=1).numpy()
    targets = test_labels.numpy()
    sr_results[seed] = evaluate_selection(preds, targets, conf, "SR")
    print(f"  Seed {seed}: AURCC={sr_results[seed]['aurcc']:.4f} "
          f"Cov@95={sr_results[seed]['cov95']:.1f}% Cov@99={sr_results[seed]['cov99']:.1f}%")

# MCD
print("\n--- MC Dropout (MCD) ---")

class BackboneClassifierMCD(nn.Module):
    def __init__(self, backbone, classifier, dropout_rate=0.3):
        super().__init__()
        self.backbone = backbone
        self.dropout = nn.Dropout(p=dropout_rate)
        self.classifier = classifier
    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(self.dropout(features)), features

mcd_results = {}
for seed in CFG["seeds"]:
    print(f"  Seed {seed}: loading model...", end=" ")
    base_model = load_trained_model(seed)
    mcd_model = BackboneClassifierMCD(base_model.backbone, base_model.classifier,
                                       CFG["mcd_dropout_rate"]).to(device)
    mcd_model.eval()
    for m in mcd_model.modules():
        if isinstance(m, nn.Dropout):
            m.train()

    all_probs, all_labels_list = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            batch_probs = torch.zeros(images.size(0), num_classes).to(device)
            for _ in range(CFG["mcd_forward_passes"]):
                logits, _ = mcd_model(images)
                batch_probs += F.softmax(logits, dim=1)
            batch_probs /= CFG["mcd_forward_passes"]
            all_probs.append(batch_probs.cpu())
            all_labels_list.append(labels)

    avg_probs = torch.cat(all_probs)
    all_labels_t = torch.cat(all_labels_list)
    preds = avg_probs.argmax(dim=1).numpy()
    entropy = -(avg_probs * torch.log(avg_probs + 1e-8)).sum(dim=1)
    conf = -entropy.numpy()
    targets = all_labels_t.numpy()

    mcd_results[seed] = evaluate_selection(preds, targets, conf, "MCD")
    print(f"AURCC={mcd_results[seed]['aurcc']:.4f} "
          f"Cov@95={mcd_results[seed]['cov95']:.1f}% Cov@99={mcd_results[seed]['cov99']:.1f}%")
    del mcd_model, base_model; torch.cuda.empty_cache()

print("\nSR + MCD complete.")

In [ ]:
# Cell 5
# AquaSelect: Selection Head (frozen) + Temperature Scaling + UDAS Fusion 

# Selection Head
class SelectionHead(nn.Module):
    def __init__(self, feature_dim, hidden_dim=256):
        super().__init__()
        self.selector = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden_dim, 64), nn.ReLU(),
            nn.Linear(64, 1), nn.Sigmoid(),
        )
    def forward(self, features):
        return self.selector(features).squeeze(-1)

class AquaSelectModel(nn.Module):
    def __init__(self, backbone, classifier, feature_dim):
        super().__init__()
        self.backbone = backbone
        self.classifier = classifier
        self.selection_head = SelectionHead(feature_dim)
        for param in self.backbone.parameters():
            param.requires_grad = False
        for param in self.classifier.parameters():
            param.requires_grad = False
    def forward(self, x):
        with torch.no_grad():
            features = self.backbone(x)
            logits = self.classifier(features)
        sel_score = self.selection_head(features.detach())
        return logits, sel_score, features.detach()

# Temperature Scaler
class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)
    def forward(self, logits):
        return logits / self.temperature
    def fit(self, val_logits, val_labels, max_iter=50, lr=0.01):
        val_logits, val_labels = val_logits.to(device), val_labels.to(device)
        self.to(device)
        nll = nn.CrossEntropyLoss()
        optimizer = torch.optim.LBFGS([self.temperature], lr=lr, max_iter=max_iter)
        def closure():
            optimizer.zero_grad()
            loss = nll(self.forward(val_logits), val_labels)
            loss.backward()
            return loss
        optimizer.step(closure)
        print(f"    Temperature: {self.temperature.item():.4f}")
        return self.temperature.item()

# UDAS features
def compute_visibility_score(image_np):
    gray = cv2.cvtColor(image_np, cv2.COLOR_RGB2GRAY)
    f_shift = np.fft.fftshift(np.fft.fft2(gray))
    magnitude = np.abs(f_shift)
    h, w = gray.shape
    cy, cx = h // 2, w // 2
    radius = min(h, w) // 8
    Y, X = np.ogrid[:h, :w]
    mask = (Y - cy)**2 + (X - cx)**2 <= radius**2
    return 1.0 - (magnitude[mask].sum() / (magnitude.sum() + 1e-8))

def compute_color_cast_score(image_np):
    means = image_np.mean(axis=(0, 1))
    return np.sqrt(((means - means.mean()) ** 2).sum())

def extract_udas(image_paths, labels, indices, transform):
    vis, cc = [], []
    for i, idx in enumerate(indices):
        img = Image.open(image_paths[idx]).convert("RGB")
        if transform: img = transform(img)
        img_np = np.array(img)
        vis.append(compute_visibility_score(img_np))
        cc.append(compute_color_cast_score(img_np))
        if (i + 1) % 500 == 0:
            print(f"    {i+1}/{len(indices)}...", end="\r")
    print(f"    {len(indices)}/{len(indices)} done.    ")
    return np.array(vis), np.array(cc)

print("Extracting UDAS features...")
print("  Val set:")
val_vis, val_cc = extract_udas(all_image_paths, all_labels, val_indices, raw_transform)
print("  Test set:")
test_vis, test_cc = extract_udas(all_image_paths, all_labels, test_indices, raw_transform)

cc_99 = np.percentile(np.concatenate([val_cc, test_cc]), 99)
val_cc_norm = np.clip(val_cc / (cc_99 + 1e-8), 0, 1)
test_cc_norm = np.clip(test_cc / (cc_99 + 1e-8), 0, 1)
print(f"  UDAS stats -- Val vis: {val_vis.mean():.3f}±{val_vis.std():.3f}, "
      f"Test vis: {test_vis.mean():.3f}±{test_vis.std():.3f}")

print("\nAquaSelect modules + UDAS ready.")

In [ ]:
# Cell 6
# Run AquaSelect: train selection head + temp scaling + UDAS fusion 

def train_selection_head(seed):
    set_seed(seed)
    base_model = load_trained_model(seed)
    model = AquaSelectModel(base_model.backbone, base_model.classifier, 768).to(device)
    
    trainable_params = list(model.selection_head.parameters())
    print(f"    Trainable: {sum(p.numel() for p in trainable_params)/1e3:.1f}K params")
    
    optimizer = AdamW(trainable_params, lr=CFG["sel_lr"], weight_decay=1e-4)
    bce = nn.BCELoss()
    best_val_loss, patience_counter, best_state = float("inf"), 0, None
    
    for epoch in range(1, CFG["sel_epochs"] + 1):
        model.selection_head.train()
        train_loss, total = 0, 0
        for images, labels in train_aug_loader:
            images, labels = images.to(device), labels.to(device)
            logits, sel_scores, _ = model(images)
            with torch.no_grad():
                binary_target = (logits.argmax(1) == labels).float()
            loss = bce(sel_scores, binary_target)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            train_loss += loss.item() * labels.size(0); total += labels.size(0)
        train_loss /= total
        
        model.eval()
        val_loss, val_total = 0, 0
        val_scores_l, val_targets_l = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                logits, sel_scores, _ = model(images)
                bt = (logits.argmax(1) == labels).float()
                val_loss += bce(sel_scores, bt).item() * labels.size(0)
                val_total += labels.size(0)
                val_scores_l.append(sel_scores.cpu()); val_targets_l.append(bt.cpu())
        val_loss /= val_total
        
        improved = ""
        if val_loss < best_val_loss:
            best_val_loss = val_loss; patience_counter = 0
            best_state = copy.deepcopy(model.selection_head.state_dict()); improved = " *"
        else:
            patience_counter += 1
        
        print(f"    Epoch {epoch}/{CFG['sel_epochs']} | Train: {train_loss:.4f} | "
              f"Val: {val_loss:.4f} | Pat: {patience_counter}/{CFG['sel_patience']}{improved}")
        if patience_counter >= CFG["sel_patience"]:
            print(f"    Early stopping at epoch {epoch}"); break
    
    model.selection_head.load_state_dict(best_state)
    model.eval()
    
    # Separation stats
    vs = torch.cat(val_scores_l).numpy(); vt = torch.cat(val_targets_l).numpy()
    correct_mask = vt == 1
    sep = vs[correct_mask].mean() - vs[~correct_mask].mean()
    print(f"    Separation: {sep:.3f} (correct: {vs[correct_mask].mean():.3f}, incorrect: {vs[~correct_mask].mean():.3f})")
    
    # Extract scores
    def get_scores(loader):
        scores = []
        with torch.no_grad():
            for images, _ in loader:
                _, s, _ = model(images.to(device))
                scores.append(s.cpu())
        return torch.cat(scores).numpy()
    
    return get_scores(val_loader), get_scores(test_loader), model


print("="*70)
print("  AQUASELECT -- Sea Animals")
print("="*70)

aq_results = {}

for seed in CFG["seeds"]:
    print(f"\n--- Seed {seed} ---")
    
    # Step 1: Selection head
    print("  [1/3] Training selection head...")
    val_sel, test_sel, model = train_selection_head(seed)
    del model; torch.cuda.empty_cache()
    
    # Step 2: Temperature scaling
    print("  [2/3] Temperature scaling...")
    temp_scaler = TemperatureScaler()
    val_logits = sa1_data["val_logits"][seed]
    val_labels = sa1_data["val_labels"]
    T = temp_scaler.fit(val_logits, val_labels)
    
    # Step 3: UDAS fusion
    print("  [3/3] UDAS fusion...")
    test_logits = sa1_data["test_logits"][seed]
    test_labels = sa1_data["test_labels"]
    
    with torch.no_grad():
        val_cal_probs = F.softmax(temp_scaler(val_logits.to(device)), dim=1).cpu()
        test_cal_probs = F.softmax(temp_scaler(test_logits.to(device)), dim=1).cpu()
    
    val_feats = np.stack([val_sel, val_cal_probs.max(1).values.numpy(), val_vis, 1.0 - val_cc_norm], axis=1)
    test_feats = np.stack([test_sel, test_cal_probs.max(1).values.numpy(), test_vis, 1.0 - test_cc_norm], axis=1)
    
    val_preds = val_logits.argmax(1).numpy()
    val_correct = (val_preds == val_labels.numpy()).astype(int)
    
    lr_model = LogisticRegression(max_iter=1000, random_state=42)
    lr_model.fit(val_feats, val_correct)
    test_conf = lr_model.predict_proba(test_feats)[:, 1]
    
    print(f"    Fusion weights: {lr_model.coef_[0]}")
    
    preds = test_logits.argmax(1).numpy()
    targets = test_labels.numpy()
    result = evaluate_selection(preds, targets, test_conf, "AquaSelect")
    result["test_confidence"] = test_conf
    aq_results[seed] = result
    
    sr_aurcc = sr_results[seed]["aurcc"]
    delta = result["aurcc"] - sr_aurcc
    print(f"  >> AURCC={result['aurcc']:.4f} | Cov@95={result['cov95']:.1f}% | Cov@99={result['cov99']:.1f}%")
    print(f"  >> vs SR: AURCC delta = {delta:+.4f} ({'better' if delta < 0 else 'worse'})")

print(f"\n{'='*70}")
print("AquaSelect complete.")

In [ ]:
# Cell 7
# Results: comparison table + selective accuracy + RC curves 

def method_summary(rd, seeds):
    aurccs = [rd[s]["aurcc"] for s in seeds]
    cov95s = [rd[s]["cov95"] for s in seeds]
    cov99s = [rd[s]["cov99"] for s in seeds]
    return np.mean(aurccs), np.std(aurccs), np.mean(cov95s), np.std(cov95s), np.mean(cov99s), np.std(cov99s)

print("="*70)
print("  SEA ANIMALS -- SELECTIVE PREDICTION RESULTS (ConvNeXt-Tiny)")
print("="*70)

for name, rd in [("SR", sr_results), ("MCD", mcd_results), ("AquaSelect", aq_results)]:
    am, astd, c95m, c95s, c99m, c99s = method_summary(rd, CFG["seeds"])
    print(f"  {name:>12} | AURCC: {am:.4f} ± {astd:.4f} | Cov@95: {c95m:.1f} ± {c95s:.1f}% | Cov@99: {c99m:.1f} ± {c99s:.1f}%")

# Per-seed detailed
print(f"\n{'='*70}")
print("  Per-Seed Results")
print("="*70)

rows = []
for name, rd in [("SR", sr_results), ("MCD", mcd_results), ("AquaSelect", aq_results)]:
    for seed in CFG["seeds"]:
        r = rd[seed]
        rows.append({"Method": name, "Seed": seed, "Full Acc (%)": f"{r['full_acc']:.2f}",
                      "AURCC": f"{r['aurcc']:.4f}", "Cov@95 (%)": f"{r['cov95']:.1f}",
                      "Cov@99 (%)": f"{r['cov99']:.1f}"})
print(pd.DataFrame(rows).to_string(index=False))

# Selective accuracy at coverage levels
print(f"\n{'='*70}")
print("  Selective Accuracy at Coverage (seed 42)")
print("="*70)

coverage_levels = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 1.00]
seed = 42
test_logits = sa1_data["test_logits"][seed]
test_labels = sa1_data["test_labels"].numpy()
test_preds = test_logits.argmax(1).numpy()
sr_conf = F.softmax(test_logits, dim=1).max(1).values.numpy()
aq_conf = aq_results[seed]["test_confidence"]

header = f"{'Method':>12} |"
for c in coverage_levels:
    header += f" {c*100:>5.0f}% |"
print(header)
print("-" * len(header))

for name, preds, conf in [("SR", test_preds, sr_conf), ("AquaSelect", test_preds, aq_conf)]:
    sorted_idx = np.argsort(-conf)
    sorted_correct = (preds == test_labels)[sorted_idx]
    row = f"{name:>12} |"
    for c in coverage_levels:
        k = max(1, int(c * len(sorted_correct)))
        acc = sorted_correct[:k].mean() * 100
        row += f" {acc:>5.1f}% |"
    print(row)

# Risk-coverage curves
fig, ax = plt.subplots(1, 1, figsize=(8, 5.5))

styles = {"SR": ("#888888", "--", 1.5), "MCD": ("#4477AA", "--", 1.5), "AquaSelect": ("#EE3377", "-", 2.5)}

for name, rd in [("SR", sr_results), ("MCD", mcd_results), ("AquaSelect", aq_results)]:
    all_risks = []
    for seed in CFG["seeds"]:
        test_logits_s = sa1_data["test_logits"][seed]
        test_labels_s = sa1_data["test_labels"].numpy()
        preds_s = test_logits_s.argmax(1).numpy()
        if name == "SR":
            conf_s = F.softmax(test_logits_s, dim=1).max(1).values.numpy()
        elif name == "AquaSelect":
            conf_s = rd[seed]["test_confidence"]
        else:
            conf_s = None  # MCD needs separate handling

        if conf_s is not None:
            _, risks, _ = compute_risk_coverage_curve(preds_s, test_labels_s, conf_s)
            all_risks.append(risks)

    if name == "MCD":
        # MCD predictions differ per seed
        for seed in CFG["seeds"]:
            # Recompute from stored result
            r = rd[seed]
            pass
        test_logits_42 = sa1_data["test_logits"][42]
        sr_preds_42 = test_logits_42.argmax(1).numpy()
        continue

    if all_risks:
        mean_risks = np.mean(all_risks, axis=0)
        coverages = np.linspace(1/len(mean_risks), 1.0, len(mean_risks))
        color, ls, lw = styles[name]
        ax.plot(coverages, mean_risks, color=color, linestyle=ls, linewidth=lw, label=name)

ax.set_xlabel("Coverage (φ)", fontsize=12)
ax.set_ylabel("Selective Risk (Error Rate)", fontsize=12)
ax.set_title("Sea Animals -- Risk-Coverage Curves (ConvNeXt-Tiny)", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.set_xlim([0, 1]); ax.set_ylim([0, 0.25])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CFG["output_dir"], "sea_animals_risk_coverage.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: sea_animals_risk_coverage.png")

In [ ]:
# Cell 8
# Save results + done 

save_data = {
    "sr_results": {s: {k: v for k, v in sr_results[s].items()} for s in CFG["seeds"]},
    "mcd_results": {s: {k: v for k, v in mcd_results[s].items()} for s in CFG["seeds"]},
    "aquaselect_results": {s: {k: v for k, v in aq_results[s].items()} for s in CFG["seeds"]},
    "label_names": label_names,
    "num_classes": num_classes,
    "config": CFG,
}

results_path = os.path.join(CFG["output_dir"], "sea_animals_selective_results.pth")
torch.save(save_data, results_path)
print(f"Saved: {results_path} ({os.path.getsize(results_path) / 1e6:.1f} MB)")

print(f"\nAll outputs in {CFG['output_dir']}:")
for f in sorted(os.listdir(CFG["output_dir"])):
    fpath = os.path.join(CFG["output_dir"], f)
    if os.path.isfile(fpath):
        print(f"  {f} ({os.path.getsize(fpath) / 1e6:.1f} MB)")

print("\nNotebook SA-2 complete.")
print("If AquaSelect AURCC < SR AURCC across all seeds: framework generalizes. ✓")

In [ ]:
# Cell 9
# Train across all seeds 

all_results = {}
total_start = time.time()

for seed in CFG["seeds"]:
    seed_start = time.time()
    best_state, history, feature_dim = train_one_seed(seed, train_loader, val_loader, CFG)
    elapsed = (time.time() - seed_start) / 60
    all_results[seed] = {
        "best_state": best_state,
        "history": history,
        "feature_dim": feature_dim,
    }
    print(f"\nSeed {seed} completed in {elapsed:.1f} min")

total_elapsed = (time.time() - total_start) / 60
print(f"\n{'='*70}")
print(f"All seeds completed in {total_elapsed:.1f} min")

In [ ]:
# Cell 10
# Clean evaluation: reload best checkpoints, extract val + test logits 

criterion = nn.CrossEntropyLoss()
seed_metrics = []
saved_outputs = {}

for seed in CFG["seeds"]:
    print(f"\n--- Clean eval for seed {seed} ---")
    
    model, feature_dim = create_model(CFG["model_name"], CFG["num_classes"])
    model.load_state_dict(all_results[seed]["best_state"])
    model = model.to(device)
    model.eval()
    
    val_loss, val_top1, val_top3, val_f1, val_logits, val_labels = evaluate(model, val_loader, criterion)
    print(f"  Val  | Loss: {val_loss:.4f} | Top1: {val_top1:.2f}% | Top3: {val_top3:.2f}% | F1: {val_f1:.2f}%")
    
    test_loss, test_top1, test_top3, test_f1, test_logits, test_labels = evaluate(model, test_loader, criterion)
    print(f"  Test | Loss: {test_loss:.4f} | Top1: {test_top1:.2f}% | Top3: {test_top3:.2f}% | F1: {test_f1:.2f}%")
    
    seed_metrics.append({
        "seed": seed,
        "val_top1": val_top1, "val_top3": val_top3, "val_f1": val_f1,
        "test_top1": test_top1, "test_top3": test_top3, "test_f1": test_f1,
    })
    
    saved_outputs[seed] = {
        "val_logits": val_logits, "val_labels": val_labels,
        "test_logits": test_logits, "test_labels": test_labels,
    }
    
    del model
    torch.cuda.empty_cache()

print("\nClean evaluation complete for all seeds.")

In [ ]:
# Cell 11
# Summary table 

df_metrics = pd.DataFrame(seed_metrics)

print("Per-seed test results:")
print(df_metrics.to_string(index=False))

print(f"\n{'='*60}")
print("ConvNeXt-Tiny on Sea Animals -- Mean +/- Std across 3 seeds:")
print(f"  Top-1 Acc: {df_metrics['test_top1'].mean():.2f} +/- {df_metrics['test_top1'].std():.2f}%")
print(f"  Top-3 Acc: {df_metrics['test_top3'].mean():.2f} +/- {df_metrics['test_top3'].std():.2f}%")
print(f"  Macro F1:  {df_metrics['test_f1'].mean():.2f} +/- {df_metrics['test_f1'].std():.2f}%")

In [ ]:
# Cell 12
# Save checkpoints + logits

output_dir = CFG["output_dir"]

# Save per-seed checkpoints
for seed in CFG["seeds"]:
    ckpt_path = os.path.join(output_dir, f"convnext_tiny_sea_animals_seed_{seed}.pth")
    torch.save({
        "model_state_dict": all_results[seed]["best_state"],
        "feature_dim": all_results[seed]["feature_dim"],
        "seed": seed,
        "model_name": CFG["model_name"],
        "num_classes": CFG["num_classes"],
    }, ckpt_path)
    print(f"Saved checkpoint: {ckpt_path} ({os.path.getsize(ckpt_path) / 1e6:.1f} MB)")

# Save all val/test logits
logits_path = os.path.join(output_dir, "sea_animals_logits.pth")
torch.save({
    "val_logits": {seed: saved_outputs[seed]["val_logits"] for seed in CFG["seeds"]},
    "val_labels": saved_outputs[CFG["seeds"][0]]["val_labels"],
    "test_logits": {seed: saved_outputs[seed]["test_logits"] for seed in CFG["seeds"]},
    "test_labels": saved_outputs[CFG["seeds"][0]]["test_labels"],
    "train_indices": train_indices,
    "val_indices": val_indices,
    "test_indices": test_indices,
    "all_image_paths": all_image_paths,
    "all_labels": all_labels,
    "label_names": class_names,
    "config": CFG,
}, logits_path)
print(f"\nSaved logits: {logits_path} ({os.path.getsize(logits_path) / 1e6:.1f} MB)")

# Save metrics
df_metrics.to_csv(os.path.join(output_dir, "sea_animals_seed_metrics.csv"), index=False)
print("Saved metrics CSV")

# List outputs
print(f"\nAll outputs in {output_dir}:")
for f in sorted(os.listdir(output_dir)):
    fpath = os.path.join(output_dir, f)
    if os.path.isfile(fpath):
        print(f"  {f} ({os.path.getsize(fpath) / 1e6:.1f} MB)")

In [ ]:
# Cell 13
print("Notebook SA-1 complete.")